# Enrollment Outreach Agent – Demo

This notebook demonstrates the benefits-team outreach agent built on top of a
trained XGBoost model.  All five required demo queries are shown, plus the
explicit **refusal** behaviour for leaky features and the **fairness guardrail**
that prevents gender / marital-status / age from appearing in explanations.

**Architecture overview**
```
User query (natural language)
        │
        ▼
  EnrollmentAgent.ask()          ← agent_router.py
        │  intent detection (regex pattern matching)
        │
   ┌────┴──────────────────────────────────────────┐
   │ predict_enrollment   rank_outreach_candidates  │
   │ lookup_region_profile  explain_prediction      │
   │ validate_raw_row                               │   agent_tools.py
   └────────────────────────────────────────────────┘
        │
   ModelConfig (single config dataclass)
        │
   _ModelState (lazy-loaded XGBoost + encoders + data)
```
No LLM API is used.  The router is a pure-Python keyword/pattern matcher.
Tool boundaries and refusal behaviour are enforced in `agent_tools.py`.


In [ ]:
import sys, os
from pathlib import Path

# Notebooks don't have __file__, so we resolve the project root from
# the notebook's actual directory on disk (works no matter where Jupyter
# was launched from).
_nb_dir = Path(os.path.abspath('')).resolve()
# If we're inside notebooks/ go one level up; otherwise stay put.
_project_root = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir
_src = str(_project_root / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

from agent_router import EnrollmentAgent
agent = EnrollmentAgent()
print('Agent ready.')
print(f'  src path : {_src}')
print(f'  project  : {_project_root}')

---
## Query 1 – Who are the top outreach priorities in the Midwest?
The agent respects `hr_outreach_capacity` from the region table unless `top_k` is
overridden.  Midwest capacity = **469**.


In [ ]:
agent.ask("Who are the top 20 outreach priorities in the Midwest this window?")

---
## Query 2 – Why is employee X predicted to enroll?
The explanation cites **only** business-safe features (salary band, employment
type, dependents, prior-year status, etc.).  Gender, marital status and age
are silently excluded even if they contributed to the model.


In [ ]:
# Pick a real employee ID from the dataset
import pandas as pd
emp_csv = str(_project_root / 'data' / 'processed' / 'employees_processed_no_leaky_features.csv')
sample_id = int(pd.read_csv(emp_csv, usecols=['employee_id']).iloc[1]['employee_id'])
print(f'Demo employee_id: {sample_id}')

agent.ask(f"Why is employee {sample_id} predicted to enroll?")

---
## Query 3 – What's the region profile for the South?
Returns stats from `region_benefit_profiles.csv` including outreach capacity,
premium costs, broker rating, and mandate level.


In [ ]:
agent.ask("What is the region profile for the South?")

---
## Query 4 – What is wrong with this raw employee row? (data validation)
Demonstrates the `validate_raw_row` tool.  The raw row below intentionally
contains:
- `legacy_propensity_score` (forbidden / leaky)
- A negative salary (implausible)
- A missing required feature (`salary_band`)


In [ ]:
bad_row = {
    'employee_id':              99999,
    'legacy_propensity_score':  0.92,   # FORBIDDEN – leakage
    'salary_clean':             -5000,  # INVALID – negative salary
    'employment_type':          'Full-time',
    'region':                   'Midwest',
    'has_dependents':           'Yes',
    'broker_channel':           'Direct',
    'tenure_years':             2.5,
    # salary_band intentionally missing
}

agent.ask("What is wrong with this raw row?", raw_row=bad_row)

---
## Query 5 – Predict enrollment for a single employee
Direct model inference call.


In [ ]:
agent.ask(f"Predict enrollment probability for employee {sample_id}")

---
## Bonus Query 6 – Outreach ranking across ALL regions
Each region is capped by its own `hr_outreach_capacity`.


In [ ]:
agent.ask("Rank outreach candidates across all regions using their capacity")

---
## Guardrail Demo 1 – Explicit Refusal for Leaky Feature
Any query that mentions `legacy_propensity_score` is **refused** with an explicit
explanation, regardless of context.


In [ ]:
agent.ask("Can you predict using the legacy_propensity_score for employee 12345?")

---
## Guardrail Demo 2 – Refusal when forbidden column is in the prediction input
Even if the query itself doesn't mention the leaky column,
passing it in `raw_row` triggers the refusal guard inside the tool.


In [ ]:
row_with_leakage = {
    'employee_id':              55555,
    'legacy_propensity_score':  0.87,   # will be caught and stripped
    'hist_enrollment_rate_region': 0.62, # also forbidden
    'salary_clean':             72000,
    'employment_type':          'Full-time',
    'region':                   'West',
    'has_dependents':           'No',
    'broker_channel':           'Employer-Sponsored',
    'tenure_years':             3.1,
    'salary_band':              3,
    'salary_per_tenure':        23225.8,
    'salary_age_ratio':         2000.0,
    'is_new_hire':              0,
    'prior_year_enrolled_clean': 0,
    'has_outreach_note':        1,
}

agent.ask("Predict enrollment for this employee", raw_row=row_with_leakage)

---
## Programmatic Access – Direct Tool Calls
The agent wrapper exposes tools directly for notebook / pipeline use.


In [ ]:
# Direct rank call – returns a DataFrame
df_ranked = agent.rank(region='Northeast', top_k=10)
print('Northeast top-10 candidates:')
print(df_ranked.to_string(index=False))

In [ ]:
# Direct predict call – returns a dict
result = agent.predict(employee_id=sample_id)
print(result)

In [ ]:
# Direct region profile
profile = agent.region_profile('West')
for k, v in profile.items():
    print(f'  {k:<40}: {v}')